In [ ]:
# ── Config ──────────────────────────────────────────────────
import os; os.makedirs('output', exist_ok=True)

MORSE_NPZ = 'output/FN1_01/prep_riem.npz'
OUT_SWC   = 'output/FN1_01/neurons_riem.swc'

# ── Riemannian metric ──
ALPHA      = 16.0
SIGMA_PERP = 1.0
GAMMA      = 0.95

# ── Tip detection ──
MIN_DIST_VOX   = 20
MIN_T_TIP      = 0.50
MAX_TIPS       = 200

# ── Path filter ──
MIN_PATH_LEN_UM = 5.0
MIN_RADIUS_UM   = 0.1
MIN_MEAN_T      = 0.10   # running mean 기반 트리밍 기준
MIN_SEG_T       = 0.01   # 단일 voxel 최솟값 — 배경 통과 트리밍
MIN_TORTUOSITY  = 1.01   # 완벽한 직선(1.0)만 제거

# ── Primary merge ──
MERGE_DIST_UM  = 8.0
MERGE_DOT_MIN  = 0.98

In [ ]:
# ── FileHFM setup ────────────────────────────────────────────
import agd
from agd import Eikonal, Metrics
import os

BIN_DIR = os.path.join(os.path.dirname(os.path.abspath('.')),
                       os.path.basename(os.getcwd()), 'bin')
# fallback: use FileHFM_binary_dir.txt if present
txt = 'FileHFM_binary_dir.txt'
if os.path.exists(txt):
    with open(txt) as f:
        BIN_DIR = f.read().strip()

agd.Eikonal.LibraryCall.binary_dir['FileHFM'] = BIN_DIR
print(f'FileHFM binaries: {BIN_DIR}')
print(f'  Riemann3: {os.path.exists(os.path.join(BIN_DIR, "FileHFM_Riemann3"))}')

In [ ]:
# ── Load ────────────────────────────────────────────────────
import numpy as np, time, gc

d              = np.load(MORSE_NPZ)
T_down         = d['T_down'].astype(np.float32)
radius_down    = d['radius_down'].astype(np.float32)
edt_down       = d['edt_down'].astype(np.float32)     # EDT 기반 실제 두께
orient_down    = d['orient_down'].astype(np.float32)
soma_mask_down = d['soma_mask_down']
voxel_down     = float(d['voxel_down'])
soma_vox_down  = d['soma_vox_down'].astype(np.float64)
soma_r_um      = float(d['soma_r_um'])

Zd, Yd, Xd = T_down.shape
print(f'T_down       : {T_down.shape}  voxel={voxel_down:.3f} µm')
print(f'orient_down  : {orient_down.shape}')
print(f'edt_down     : min={edt_down.min():.3f}  median={np.median(edt_down[edt_down>0]):.3f}'
      f'  max={edt_down.max():.3f} µm')
print(f'soma_mask    : {soma_mask_down.sum():,} voxels')

In [ ]:
# ── Build Riemannian metric M(x) ────────────────────────────
# σ_parallel(x) = σ_perp · (1 - γ·T)   ← T-adaptive anisotropy
#   T=0 (background) : σ_parallel = σ_perp      → isotropic
#   T=1 (strong tube): σ_parallel = σ_perp·(1-γ) → max anisotropy
#
# M(x) = cost² · [σ_perp·I + (σ_parallel - σ_perp)·v⊗v]
# Eikonal: sqrt(∇u^T M ∇u) = 1

t0 = time.time()

T64     = T_down.astype(np.float64)
cost2   = np.exp(-ALPHA * T64) ** 2              # (Z,Y,X)
sigma_p = SIGMA_PERP * (1.0 - GAMMA * T64)       # (Z,Y,X) adaptive
v       = orient_down.astype(np.float64)          # (Z,Y,X,3)

M = np.zeros((3, 3, Zd, Yd, Xd), dtype=np.float64)
for i in range(3):
    M[i, i] += SIGMA_PERP
    for j in range(3):
        M[i, j] += (sigma_p - SIGMA_PERP) * v[..., i] * v[..., j]

M *= cost2[np.newaxis, np.newaxis, ...]
del cost2, v, sigma_p, T64; gc.collect()

sp_min = SIGMA_PERP * (1.0 - GAMMA * float(T_down.max()))
print(f'M built in {time.time()-t0:.1f}s  shape={M.shape}  mem={M.nbytes/1e9:.2f} GB')
print(f'σ_parallel range : {sp_min:.4f} – {SIGMA_PERP:.4f}')
print(f'Anisotropy ratio  : {SIGMA_PERP/sp_min:.1f}:1  (at T=max)')

In [ ]:
# ── Riemannian Fast Marching (agd Riemann3) ─────────────────
t0 = time.time()

soma_seeds = np.argwhere(soma_mask_down).astype(np.float64)   # (N,3) z,y,x
print(f'Seeds: {len(soma_seeds):,} soma voxels')

metric = Metrics.Riemann(M)
del M; gc.collect()

hfm = Eikonal.dictIn({
    'model':        'Riemann3',
    'dims':         np.array([Zd, Yd, Xd]),
    'gridScale':    1.0,
    'metric':       metric,
    'seeds':        soma_seeds,
    'exportValues': True,
    'verbosity':    1,
})

out          = hfm.Run()
geodesic_dist = out['values'].astype(np.float32)

print(f'FMM done in {time.time()-t0:.1f}s')
print(f'Reachable : {np.isfinite(geodesic_dist).sum():,}')
print(f'Geo range : {geodesic_dist[np.isfinite(geodesic_dist)].min():.3f} – '
      f'{geodesic_dist[np.isfinite(geodesic_dist)].max():.3f}')

In [ ]:
# ── Tip detection: T_down local maxima ──────────────────────
import matplotlib.pyplot as plt
from skimage.feature import peak_local_max

_peaks = peak_local_max(
    T_down,
    min_distance   = MIN_DIST_VOX,
    threshold_abs  = MIN_T_TIP,
    exclude_border = False,
)
tip_coords_all = _peaks if _peaks.dtype != bool else np.argwhere(_peaks)

tip_vals = T_down[tip_coords_all[:,0], tip_coords_all[:,1], tip_coords_all[:,2]]
sort_idx     = np.argsort(tip_vals)[::-1]
tip_coords_s = tip_coords_all[sort_idx][:MAX_TIPS]
tip_vals_s   = tip_vals[sort_idx][:MAX_TIPS]

# filter: reachable tips only
reachable = np.isfinite(geodesic_dist[tip_coords_s[:,0],
                                      tip_coords_s[:,1],
                                      tip_coords_s[:,2]])
tip_coords_s = tip_coords_s[reachable]
tip_vals_s   = tip_vals_s[reachable]

print(f'Tips detected : {len(tip_coords_all):,}')
print(f'Tips selected : {len(tip_coords_s)} (reachable, top-{MAX_TIPS} by T)')
print(f'T range       : {tip_vals_s[-1]:.3f} – {tip_vals_s[0]:.3f}')

geo_finite = geodesic_dist.copy()
geo_finite[~np.isfinite(geo_finite)] = 0

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
axes[0].imshow(T_down.max(axis=0), cmap='hot', vmin=0, vmax=1, origin='upper')
axes[0].scatter(tip_coords_s[:,2], tip_coords_s[:,1],
                s=5, c=tip_vals_s, cmap='cool', alpha=0.8)
axes[0].scatter([soma_vox_down[2]], [soma_vox_down[1]],
                c='red', s=80, marker='*', zorder=5)
axes[0].set_title(f'T_down + tips (n={len(tip_coords_s)})'); axes[0].axis('off')

axes[1].imshow((geo_finite / (geo_finite.max()+1e-8)).max(axis=0),
               cmap='viridis_r', origin='upper')
axes[1].scatter(tip_coords_s[:,2], tip_coords_s[:,1], s=5, c='red', alpha=0.7)
axes[1].set_title('Riemannian geodesic dist MIP'); axes[1].axis('off')
plt.show()

In [ ]:
# ── Traceback: discrete gradient descent + distal trimming ──
t0 = time.time()

def traceback_discrete(tip_vox, geo_dist, soma_mask, Zd, Yd, Xd, max_steps=200000):
    cur  = (int(tip_vox[0]), int(tip_vox[1]), int(tip_vox[2]))
    path = []
    for _ in range(max_steps):
        path.append(cur)
        if soma_mask[cur]:
            break
        z, y, x  = cur
        best_val = geo_dist[cur]
        best_nb  = None
        for dz in range(-1, 2):
            for dy in range(-1, 2):
                for dx in range(-1, 2):
                    if dz == dy == dx == 0:
                        continue
                    nz, ny, nx = z+dz, y+dy, x+dx
                    if 0 <= nz < Zd and 0 <= ny < Yd and 0 <= nx < Xd:
                        v = geo_dist[nz, ny, nx]
                        if v < best_val:
                            best_val, best_nb = v, (nz, ny, nx)
        if best_nb is None:
            break
        cur = best_nb
    return path[::-1]   # soma→tip

def path_length_um(path, voxel):
    if len(path) < 2: return 0.0
    arr = np.array(path, dtype=np.float32)
    return float(np.linalg.norm(np.diff(arr, axis=0), axis=1).sum()) * voxel

all_paths = {}
n_short = n_trimmed = n_too_short_trim = n_straight = 0

for i, tip in enumerate(tip_coords_s):
    key = (int(tip[0]), int(tip[1]), int(tip[2]))
    if not np.isfinite(geodesic_dist[key]):
        continue
    path = traceback_discrete(key, geodesic_dist, soma_mask_down, Zd, Yd, Xd)

    if path_length_um(path, voxel_down) < MIN_PATH_LEN_UM:
        n_short += 1; continue

    t_vals = np.array([T_down[k] for k in path], dtype=np.float32)
    trimmed = False
    half = len(path) // 2

    # ── MIN_SEG_T: distal 구간 첫 배경 voxel에서 트리밍 ─────
    bad = np.where(t_vals[half:] < MIN_SEG_T)[0]
    if len(bad) > 0:
        path   = path[:half + bad[0]]
        t_vals = t_vals[:half + bad[0]]
        trimmed = True
        if path_length_um(path, voxel_down) < MIN_PATH_LEN_UM:
            n_too_short_trim += 1; continue

    # ── MIN_MEAN_T: distal 구간 running mean 트리밍 ──────────
    if len(t_vals) > half:
        rm_distal = np.cumsum(t_vals[half:]) / np.arange(1, len(t_vals)-half+1)
        bad_mean  = np.where(rm_distal < MIN_MEAN_T)[0]
        if len(bad_mean) > 0:
            path   = path[:half + bad_mean[0]]
            t_vals = t_vals[:half + bad_mean[0]]
            trimmed = True
            if path_length_um(path, voxel_down) < MIN_PATH_LEN_UM:
                n_too_short_trim += 1; continue

    if trimmed:
        n_trimmed += 1

    # ── Tortuosity ────────────────────────────────────────────
    path_len_um = path_length_um(path, voxel_down)
    euclid_um   = float(np.linalg.norm(
        np.array(path[-1], dtype=np.float32) -
        np.array(path[0],  dtype=np.float32))) * voxel_down
    if path_len_um / (euclid_um + 1e-8) < MIN_TORTUOSITY:
        n_straight += 1; continue

    all_paths[i] = path

print(f'Traceback : {time.time()-t0:.1f}s')
print(f'Paths kept: {len(all_paths)}')
print(f'  trimmed & kept       : {n_trimmed}')
print(f'  filtered — short     : {n_short}')
print(f'  too_short_after_trim : {n_too_short_trim}')
print(f'  straight             : {n_straight}')
if all_paths:
    lens = [path_length_um(p, voxel_down) for p in all_paths.values()]
    print(f'  Length: min={min(lens):.1f}  max={max(lens):.1f}  mean={np.mean(lens):.1f} µm')

In [ ]:
# ── Tree 구성 (shared trunk deduplication) ──────────────────
sc = soma_vox_down
soma_x = sc[2]*voxel_down; soma_y = sc[1]*voxel_down; soma_z = sc[0]*voxel_down

swc_rows    = [(1, 1, soma_x, soma_y, soma_z, soma_r_um, -1)]
node_id_map = {}
next_id     = 2

sorted_keys = sorted(all_paths.keys(),
                     key=lambda k: float(geo_finite[tip_coords_s[k][0],
                                                    tip_coords_s[k][1],
                                                    tip_coords_s[k][2]]),
                     reverse=False)

for bi in sorted_keys:
    prev_swc_id = 1
    for key in all_paths[bi]:
        if soma_mask_down[key]:
            node_id_map[key] = 1
            prev_swc_id = 1
            continue
        if key in node_id_map:
            prev_swc_id = node_id_map[key]
            continue
        z, y, x = key
        r = max(float(edt_down[z, y, x]), MIN_RADIUS_UM)   # EDT 기반 반경
        swc_rows.append((next_id, 3,
                         x*voxel_down, y*voxel_down, z*voxel_down,
                         r, prev_swc_id))
        node_id_map[key] = next_id
        prev_swc_id      = next_id
        next_id         += 1

print(f'SWC nodes: {next_id-1:,}  (branches: {len(all_paths)})')
swc_rows_dict = {r[0]: r for r in swc_rows}

In [ ]:
# ── Primary branch 병합 ──────────────────────────────────────
from scipy.spatial import cKDTree
from collections import defaultdict

primary_nodes = {nid: swc_rows_dict[nid]
                 for nid in [r[0] for r in swc_rows if r[6] == 1]}

def branch_dir(nid):
    n = swc_rows_dict[nid]; sc_ = swc_rows_dict[1]
    v = np.array([n[2]-sc_[2], n[3]-sc_[3], n[4]-sc_[4]])
    return v / (np.linalg.norm(v) + 1e-8)

def node_pos(nid):
    n = swc_rows_dict[nid]; return np.array([n[2], n[3], n[4]])

pids   = list(primary_nodes.keys())
p_pos  = np.array([node_pos(p) for p in pids])
p_dirs = np.array([branch_dir(p) for p in pids])

parent_uf = {p: p for p in pids}
def find(x):
    while parent_uf[x] != x: parent_uf[x] = parent_uf[parent_uf[x]]; x = parent_uf[x]
    return x
def union(a, b): parent_uf[find(a)] = find(b)

if len(pids) > 1:
    tree = cKDTree(p_pos)
    for i, j in tree.query_pairs(MERGE_DIST_UM):
        if abs(float(p_dirs[i] @ p_dirs[j])) >= MERGE_DOT_MIN:
            union(pids[i], pids[j])

groups = defaultdict(list)
for p in pids: groups[find(p)].append(p)

def subtree_tip_count(nid):
    count, stack = 0, [nid]
    while stack:
        cur = stack.pop()
        kids = [r[0] for r in swc_rows if r[6] == cur]
        if not kids: count += 1
        stack.extend(kids)
    return count

to_remove, merged_count = set(), 0
for rep, members in groups.items():
    if len(members) == 1: continue
    best = max(members, key=subtree_tip_count)
    for m in members:
        if m != best: to_remove.add(m)
    merged_count += len(members) - 1

def get_subtree_ids(root):
    result, stack = set(), [root]
    while stack:
        cur = stack.pop(); result.add(cur)
        stack.extend([r[0] for r in swc_rows if r[6] == cur])
    return result

remove_ids = set()
for nid in to_remove: remove_ids |= get_subtree_ids(nid)
swc_rows = [r for r in swc_rows if r[0] not in remove_ids]

print(f'Primary before merge : {len(pids)}')
print(f'Merged (removed)     : {merged_count}')
print(f'SWC nodes after merge: {len(swc_rows):,}')

In [ ]:
# ── SWC 저장 ─────────────────────────────────────────────────
header = [
    f'# tracer_aniso — Riemannian FMM  α={ALPHA}',
    f'# GAMMA={GAMMA}  SIGMA_PERP={SIGMA_PERP}  ALPHA={ALPHA}',
    f'# tips={len(tip_coords_s)}  paths={len(all_paths)}  nodes={next_id-1}',
    '# id type x y z radius parent',
]
lines = header + [
    f'{r[0]} {r[1]} {r[2]:.4f} {r[3]:.4f} {r[4]:.4f} {r[5]:.4f} {r[6]}'
    for r in swc_rows
]
with open(OUT_SWC, 'w') as f:
    f.write('\n'.join(lines) + '\n')
print(f'Saved: {OUT_SWC}')

In [ ]:
# ── CHECK: SWC 시각화 ───────────────────────────────────────
import matplotlib.pyplot as plt

with open(OUT_SWC) as f:
    lines = [l for l in f if not l.startswith('#') and l.strip()]

nodes = np.array([list(map(float, l.split()[2:5])) for l in lines])
types = np.array([int(l.split()[1]) for l in lines])

den = nodes[types == 3]
som = nodes[types == 1]

print(f'SWC nodes  : {len(nodes):,}')
print(f'X: {nodes[:,0].min():.0f}–{nodes[:,0].max():.0f}  '
      f'Y: {nodes[:,1].min():.0f}–{nodes[:,1].max():.0f}  '
      f'Z: {nodes[:,2].min():.0f}–{nodes[:,2].max():.0f} µm')

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
for ax, (xi, yi, lbl) in zip(axes, [(0,1,'XY'), (0,2,'XZ')]):
    ax.scatter(den[:,xi], den[:,yi], s=0.3, alpha=0.15,
               color='black', linewidths=1)
    ax.scatter(som[:,xi], som[:,yi], s=60, color='red',
               marker='*', zorder=5, label='soma')
    ax.set_xlabel('X (µm)'); ax.set_ylabel('Y/Z (µm)')
    ax.set_title(f'SWC Riemannian — {lbl}')
    ax.set_aspect('equal'); ax.legend()
plt.show()